# Fluxo 2 — Telemetria e Eventos Web (Schema Drift)

**Universidade Mackenzie — MBA Engenharia de Dados**  
Disciplina: Data Collect and Storage | Prof. Filipe Quintieri Lima  
Aluno: Matheus Alves da Silva | RA: 10752559

| Item | Valor |
|---|---|
| Estratégia | Incremental (Insert-Only) |
| Formato de saída | JSONL (JSON Lines) |
| Destino | MinIO — camada Bronze |
| Watermark | Lido do último metadata com `status: success` |

In [1]:
# Importações e variáveis de configuração
import json
import os
from datetime import date, datetime

import boto3
from botocore.client import Config
from botocore.exceptions import ClientError
from dotenv import load_dotenv
from pyspark.sql import SparkSession

load_dotenv()

PG_HOST     = os.getenv("PG_HOST")
PG_PORT     = int(os.getenv("PG_PORT", "5432"))
PG_DB       = os.getenv("PG_DB")
PG_USER     = os.getenv("PG_USER")
PG_PASSWORD = os.getenv("PG_PASSWORD")
PG_JDBC_URL = f"jdbc:postgresql://{PG_HOST}:{PG_PORT}/{PG_DB}"

MINIO_ENDPOINT   = os.getenv("MINIO_ENDPOINT")
MINIO_ACCESS_KEY = os.getenv("MINIO_ACCESS_KEY")
MINIO_SECRET_KEY = os.getenv("MINIO_SECRET_KEY")
MINIO_BUCKET     = os.getenv("MINIO_BUCKET", "datalake")

METADATA_PREFIX   = "bronze/telemetria/metadata/"
WATERMARK_DEFAULT = os.getenv("ULTIMA_EXECUCAO", "2000-01-01 00:00:00")

DATA_INGESTAO     = date.today().strftime("%Y-%m-%d")
TIMESTAMP_INGESTAO = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

OUTPUT_PATH = (
    f"s3a://{MINIO_BUCKET}/bronze/telemetria/eventos_web/"
    f"data_ingestao={DATA_INGESTAO}/eventos.jsonl"
)

print(f"JDBC URL  : {PG_JDBC_URL}")
print(f"MinIO     : {MINIO_ENDPOINT}")
print(f"Output    : {OUTPUT_PATH}")

JDBC URL  : jdbc:postgresql://postgres:5432/postgres
MinIO     : http://minio:9000
Output    : s3a://datalake/bronze/telemetria/eventos_web/data_ingestao=2026-05-30/eventos.jsonl


In [2]:
# Funções de Metadata e Watermark (boto3 — sem Spark)

def get_minio_client():
    return boto3.client(
        "s3",
        endpoint_url=MINIO_ENDPOINT,
        aws_access_key_id=MINIO_ACCESS_KEY,
        aws_secret_access_key=MINIO_SECRET_KEY,
        config=Config(signature_version="s3v4"),
        region_name="us-east-1",
    )


def ler_watermark(s3_client) -> str:
    """Lista os arquivos de metadata e retorna o timestamp do mais recente com status 'success'."""
    try:
        response = s3_client.list_objects_v2(Bucket=MINIO_BUCKET, Prefix=METADATA_PREFIX)
        objetos = response.get("Contents", [])
        if not objetos:
            print("[INFO] Nenhum metadata encontrado. Usando valor padrão.")
            return WATERMARK_DEFAULT

        for obj in sorted(objetos, key=lambda o: o["LastModified"], reverse=True):
            body = s3_client.get_object(Bucket=MINIO_BUCKET, Key=obj["Key"])
            data = json.loads(body["Body"].read().decode("utf-8"))
            for registro in (data if isinstance(data, list) else [data]):
                if registro.get("status") == "success":
                    print(f"[INFO] Watermark encontrado em '{obj['Key']}': {registro['timestamp']}")
                    return registro["timestamp"]

        print("[WARN] Nenhum metadata com status 'success'. Usando valor padrão.")
    except ClientError as e:
        print(f"[WARN] Erro ao listar metadata: {e}. Usando valor padrão.")
    return WATERMARK_DEFAULT


def upload_metadata(s3_client, registro: dict):
    key = f"{METADATA_PREFIX}{TIMESTAMP_INGESTAO}.json"
    s3_client.put_object(
        Bucket=MINIO_BUCKET,
        Key=key,
        Body=json.dumps(registro).encode("utf-8"),
        ContentType="application/json",
    )
    print(f"[OK] Metadata salvo: s3://{MINIO_BUCKET}/{key}")


print("Funções de metadata definidas.")

Funções de metadata definidas.


In [3]:
# Leitura do watermark antes de inicializar o Spark
s3 = get_minio_client()
ultima_execucao = ler_watermark(s3)

print("=" * 60)
print("FLUXO 2 — Telemetria / Eventos Web (Insert-Only / Schema Drift)")
print(f"Data de ingestão   : {DATA_INGESTAO}")
print(f"Watermark utilizado: {ultima_execucao}")
print("=" * 60)

[WARN] Nenhum metadata com status 'success'. Usando valor padrão.
FLUXO 2 — Telemetria / Eventos Web (Insert-Only / Schema Drift)
Data de ingestão   : 2026-05-30
Watermark utilizado: 2000-01-01 00:00:00


In [4]:
# Inicialização da SparkSession com JARs via Maven
spark = (
    SparkSession.builder
    .appName("Fluxo2_Telemetria_EventosWeb")
    .config(
        "spark.jars.packages",
        "org.postgresql:postgresql:42.7.2,"
        "org.apache.hadoop:hadoop-aws:3.3.4,"
        "com.amazonaws:aws-java-sdk-bundle:1.12.262",
    )
    .config("spark.hadoop.fs.s3a.endpoint", MINIO_ENDPOINT)
    .config("spark.hadoop.fs.s3a.access.key", MINIO_ACCESS_KEY)
    .config("spark.hadoop.fs.s3a.secret.key", MINIO_SECRET_KEY)
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print(f"SparkSession ativa. Versão: {spark.version}")

SparkSession ativa. Versão: 3.5.0


In [6]:
# Extração — eventos_web via JDBC
# payload (JSONB) é lido como TEXT para preservar schema drift na camada Bronze
query = (
    f"(SELECT id_evento, tipo_evento, data_criacao, CAST(payload AS TEXT) AS payload "
    f"FROM eventos_web WHERE data_criacao > '{ultima_execucao}') AS t"
)

print(f"[INFO] Extraindo eventos_web com data_criacao > {ultima_execucao}")

df_eventos = spark.read.jdbc(
    url=PG_JDBC_URL,
    table=query,
    properties={
        "user": PG_USER,
        "password": PG_PASSWORD,
        "driver": "org.postgresql.Driver",
    },
)

print(f"[INFO] {df_eventos.count()} evento(s) extraído(s) de 'eventos_web'.")
df_eventos.printSchema()

[INFO] Extraindo eventos_web com data_criacao > 2000-01-01 00:00:00
[INFO] 50000 evento(s) extraído(s) de 'eventos_web'.
root
 |-- id_evento: string (nullable = true)
 |-- tipo_evento: string (nullable = true)
 |-- data_criacao: timestamp (nullable = true)
 |-- payload: string (nullable = true)



In [7]:
# Carga — Bronze MinIO em formato JSONL
try:
    if df_eventos.rdd.isEmpty():
        print("[INFO] Nenhum evento novo. Nenhum arquivo gerado.")
    else:
        (
            df_eventos
            .coalesce(1)
            .write
            .mode("overwrite")
            .json(OUTPUT_PATH)
        )
        print(f"[OK] Arquivo JSONL gravado em: {OUTPUT_PATH}")

    upload_metadata(s3, {"timestamp": TIMESTAMP_INGESTAO, "status": "success"})
    print("FLUXO 2 concluído com sucesso.")

except Exception as e:
    print(f"[ERROR] Ocorreu um erro: {e}")
    upload_metadata(s3, {"timestamp": TIMESTAMP_INGESTAO, "status": "error", "mensagem": str(e)})
    raise

[OK] Arquivo JSONL gravado em: s3a://datalake/bronze/telemetria/eventos_web/data_ingestao=2026-05-30/eventos.jsonl
[OK] Metadata salvo: s3://datalake/bronze/telemetria/metadata/2026-05-30 15:00:07.json
FLUXO 2 concluído com sucesso.


In [8]:
# Encerramento da SparkSession
spark.stop()
print("SparkSession encerrada.")

SparkSession encerrada.
